# AEGIS AI — The Hidden Risk Intelligence Engine
### An end-to-end machine learning laboratory for detecting, explaining, and anticipating emerging risk

**Project type:** Applied machine learning • Predictive analytics • Anomaly detection • Explainable AI  
**Notebook mode:** Fully self-contained; creates a reproducible synthetic dataset.

> **Important:** This notebook uses simulated data. Its scores and findings demonstrate a workflow, not real-world risk prevalence or a validated operational decision system.

## 1. Problem framing

Many systems do not fail suddenly. Small signals accumulate: service delays, resource pressure, unusual patterns, and repeated near-misses. A useful intelligence workflow should:

1. Detect records with elevated risk.
2. Identify unusual observations that do not resemble the majority.
3. Explain which factors contribute most to a model's predictions.
4. Explore how changing conditions could alter predicted risk.

**Core question:** Can we combine supervised prediction, unsupervised anomaly detection, and scenario analysis into one transparent decision-support prototype?

In [ ]:
# 2. Imports and reproducibility
import warnings, sys, subprocess, importlib.util
warnings.filterwarnings("ignore")

required = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
print("Environment ready. Random seed:", RANDOM_STATE)

## 3. Generate a realistic synthetic signal dataset

Each row represents one fictional service-area observation over a period. Features describe demand, access, capacity, operational friction, and environmental context.

The target is generated from a known risk mechanism with noise. This lets us test whether models can recover useful patterns. Because the data are simulated, the target mechanism is known by construction.

In [ ]:
N = 6500
regions = rng.choice(["North", "South", "East", "West", "Central"], N,
                     p=[.19, .23, .18, .20, .20])
period = rng.integers(1, 25, N)

population_pressure = np.clip(rng.normal(0.52, 0.19, N), 0, 1)
travel_time = np.clip(rng.gamma(2.2, 17, N), 2, 150)
capacity_utilization = np.clip(rng.beta(3.2, 2.0, N), 0.05, 1.35)
staff_shortage = np.clip(rng.beta(2.0, 5.0, N), 0, 1)
resource_gap = np.clip(rng.beta(2.2, 3.4, N), 0, 1)
service_delay = np.clip(rng.gamma(2.0, 1.8, N), 0, 30)
repeat_incidents = rng.poisson(1.4, N)
weather_disruption = rng.binomial(1, .13, N)
digital_access = np.clip(rng.beta(4, 2.8, N), 0, 1)
community_support = np.clip(rng.beta(3.2, 2.5, N), 0, 1)
trend = (period - 1) / 23

region_effect = pd.Series(regions).map({
    "North": .12, "South": -.04, "East": .08, "West": -.02, "Central": .03
}).to_numpy()

# Latent risk mechanism: nonlinear interactions + noise
logit = (
    -3.0
    + 1.55 * population_pressure
    + 1.30 * (capacity_utilization - .55)
    + 1.25 * staff_shortage
    + 1.15 * resource_gap
    + .025 * travel_time
    + .13 * service_delay
    + .28 * repeat_incidents
    + .62 * weather_disruption
    - .95 * digital_access
    - .65 * community_support
    + .75 * population_pressure * capacity_utilization
    + .55 * staff_shortage * resource_gap
    + .38 * trend
    + region_effect
    + rng.normal(0, .55, N)
)
probability = 1 / (1 + np.exp(-logit))
risk_event = rng.binomial(1, probability)

df = pd.DataFrame({
    "region": regions, "period": period,
    "population_pressure": population_pressure,
    "travel_time_min": travel_time,
    "capacity_utilization": capacity_utilization,
    "staff_shortage_index": staff_shortage,
    "resource_gap_index": resource_gap,
    "service_delay_hours": service_delay,
    "repeat_incidents_90d": repeat_incidents,
    "weather_disruption": weather_disruption,
    "digital_access_index": digital_access,
    "community_support_index": community_support,
    "risk_event": risk_event
})

# Introduce realistic missingness in a few measurements
for col, frac in [("travel_time_min", .035), ("digital_access_index", .025),
                  ("community_support_index", .02), ("service_delay_hours", .03)]:
    idx = rng.choice(df.index, int(N * frac), replace=False)
    df.loc[idx, col] = np.nan

print("Dataset shape:", df.shape)
display(df.head())

## 4. Data quality and target audit

We inspect missingness, duplicates, feature ranges, and target balance before training. This is essential: model performance is only meaningful when the data pipeline is understood.

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(audit)
print("Duplicate rows:", df.duplicated().sum())
print("Target distribution:")
display(df["risk_event"].value_counts().rename("count").to_frame())
print("Positive class rate:", round(df["risk_event"].mean() * 100, 2), "%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="risk_event", ax=axes[0])
axes[0].set_title("Target class distribution")
axes[0].set_xlabel("Risk event (0 = no, 1 = yes)")
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].plot(kind="bar", ax=axes[1])
axes[1].set_title("Missing-value rate by feature")
axes[1].set_ylabel("Fraction missing")
plt.tight_layout()
plt.show()

## 5. Exploratory data analysis

The following views examine distributions, relationships, regional differences, and risk rates across selected operational factors. Correlation is descriptive, not proof of causation.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Feature correlation map")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(data=df, x="travel_time_min", hue="risk_event", bins=35,
             stat="density", common_norm=False, element="step", ax=axes[0,0])
sns.boxplot(data=df, x="risk_event", y="capacity_utilization", ax=axes[0,1])
sns.boxplot(data=df, x="risk_event", y="staff_shortage_index", ax=axes[1,0])
sns.boxplot(data=df, x="risk_event", y="service_delay_hours", ax=axes[1,1])
axes[0,0].set_title("Travel time distribution by target")
axes[0,1].set_title("Capacity utilization by target")
axes[1,0].set_title("Staff shortage by target")
axes[1,1].set_title("Service delay by target")
plt.tight_layout()
plt.show()

In [ ]:
regional = df.groupby("region", observed=True).agg(
    observations=("risk_event", "size"),
    risk_rate=("risk_event", "mean"),
    avg_travel=("travel_time_min", "mean"),
    avg_capacity=("capacity_utilization", "mean")
).sort_values("risk_rate", ascending=False)
display(regional.style.format({
    "risk_rate": "{:.1%}", "avg_travel": "{:.1f}", "avg_capacity": "{:.2f}"
}))
ax = regional["risk_rate"].sort_values().plot(kind="barh", figsize=(9,4))
ax.set_title("Simulated risk-event rate by region")
ax.set_xlabel("Risk-event rate")
plt.tight_layout()
plt.show()

## 6. Feature engineering and leakage-safe split

We create a small number of interpretable features. The split is performed before fitting imputers, encoders, or scalers; those transformations are learned only from the training data through scikit-learn pipelines.

In [ ]:
model_df = df.copy()
model_df["pressure_capacity_interaction"] = (
    model_df["population_pressure"] * model_df["capacity_utilization"]
)
model_df["staff_resource_interaction"] = (
    model_df["staff_shortage_index"] * model_df["resource_gap_index"]
)
model_df["high_repeat_signal"] = (
    model_df["repeat_incidents_90d"] >= 3
).astype(int)

X = model_df.drop(columns="risk_event")
y = model_df["risk_event"]

categorical_features = ["region"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.22, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 3),
      "| Test positive rate:", round(y_test.mean(), 3))

## 7. Train and compare multiple models

We compare interpretable logistic regression with tree-based models. The preprocessing pipeline imputes missing values and scales numeric inputs for logistic regression; categorical region labels are one-hot encoded.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, min_samples_leaf=3, max_features="sqrt",
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=180, learning_rate=.08, max_leaf_nodes=20,
        l2_regularization=1.0, random_state=RANDOM_STATE
    )
}
pipelines = {
    name: Pipeline([("preprocess", preprocess), ("model", estimator)])
    for name, estimator in models.items()
}

In [ ]:
results = []
fitted = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    score = pipe.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, score),
        "pr_auc": average_precision_score(y_test, score)
    })
metrics = pd.DataFrame(results).set_index("model").sort_values("roc_auc", ascending=False)
display(metrics.style.format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, pipe in fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC curves")
axes[1].set_title("Precision-recall curves")
plt.tight_layout()
plt.show()

## 8. Evaluate the leading model

The highest ROC-AUC model is selected for this demonstration. We inspect its confusion matrix and classification report. The default 0.50 threshold is not automatically the best operational threshold; threshold choice depends on the cost of false alarms versus missed events.

In [ ]:
best_name = metrics.index[0]
best_model = fitted[best_name]
best_prob = best_model.predict_proba(X_test)[:, 1]
best_pred = (best_prob >= .50).astype(int)

print("Selected model:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.title("Confusion matrix — threshold 0.50")
plt.tight_layout()
plt.show()

## 9. Threshold analysis: make the trade-off visible

A risk score becomes an action only after a threshold is chosen. We inspect precision, recall, and F1 across candidate thresholds rather than assuming 0.50 is universally appropriate.

In [ ]:
thresholds = np.linspace(.10, .90, 33)
threshold_rows = []
for t in thresholds:
    p = (best_prob >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_test, p, zero_division=0),
        "recall": recall_score(y_test, p, zero_division=0),
        "f1": f1_score(y_test, p, zero_division=0)
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.iloc[::4].round(3))
threshold_df.set_index("threshold")[["precision", "recall", "f1"]].plot(figsize=(10,5))
plt.title("Threshold trade-off")
plt.ylabel("Metric")
plt.ylim(0, 1.02)
plt.tight_layout()
plt.show()

## 10. Unsupervised anomaly detection

Anomaly detection asks a different question from classification: which observations look unusual compared with the broader feature distribution? An unusual record is not necessarily harmful, and a normal-looking record is not guaranteed to be safe.

We fit Isolation Forest on the training set after applying the preprocessing pipeline.

In [ ]:
X_train_trans = best_model.named_steps["preprocess"].transform(X_train)
X_test_trans = best_model.named_steps["preprocess"].transform(X_test)

iso = IsolationForest(
    n_estimators=300, contamination="auto",
    random_state=RANDOM_STATE, n_jobs=-1
)
iso.fit(X_train_trans)
anomaly_flag = (iso.predict(X_test_trans) == -1).astype(int)
anomaly_score = -iso.decision_function(X_test_trans)

anomaly_summary = pd.DataFrame({
    "anomaly_flag": anomaly_flag,
    "predicted_risk_probability": best_prob,
    "actual_risk_event": y_test.to_numpy()
})
display(anomaly_summary.groupby("anomaly_flag").agg(
    records=("actual_risk_event", "size"),
    actual_event_rate=("actual_risk_event", "mean"),
    mean_predicted_risk=("predicted_risk_probability", "mean"),
    mean_anomaly_score=("predicted_risk_probability", "size")
).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.scatterplot(
    x=best_prob, y=anomaly_score, hue=anomaly_flag,
    palette={0: "steelblue", 1: "crimson"}, alpha=.65, ax=ax
)
ax.set_xlabel("Supervised predicted risk probability")
ax.set_ylabel("Isolation Forest anomaly score (higher = more unusual)")
ax.set_title("Risk prediction versus unusualness")
plt.tight_layout()
plt.show()

## 11. Explainability with permutation importance

Permutation importance measures how much a fitted model's score degrades when one feature is shuffled. It is model-agnostic and can reveal which inputs matter most to predictive performance on the held-out set.

It does **not** establish causality, and correlated features can share or mask importance.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc",
    n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance.head(12).round(4))
top_imp = importance.head(12).sort_values("importance_mean")
plt.figure(figsize=(10,6))
plt.barh(top_imp["feature"], top_imp["importance_mean"],
         xerr=top_imp["importance_std"])
plt.title("Permutation feature importance (ROC-AUC decrease)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 12. Local explanation: inspect one example

For one test observation, we compare its feature values with the overall training-set median. The explanation below is intentionally descriptive: it does not claim that any single feature caused the outcome.

In [ ]:
row_pos = int(np.argmax(best_prob))
sample = X_test.iloc[[row_pos]].copy()
sample_probability = float(best_prob[row_pos])
comparison = pd.DataFrame({
    "selected_record": sample.iloc[0],
    "training_median": X_train.median(numeric_only=True)
})
display(comparison)
print(f"Selected record predicted risk probability: {sample_probability:.1%}")
print("Actual label (held-out synthetic target):", int(y_test.iloc[row_pos]))

## 13. Scenario simulator

We now create a copy of one record and vary selected operational inputs. The model estimates how the predicted score changes under those hypothetical feature values.

**Caution:** These are model-based what-if scenarios, not causal treatment effects. The simulator assumes other inputs remain fixed and may explore combinations rarely seen during training.

In [ ]:
base = X_test.iloc[[row_pos]].copy()

def scenario_prediction(record, changes):
    altered = record.copy()
    for feature, value in changes.items():
        altered.loc[:, feature] = value
    return float(best_model.predict_proba(altered)[:, 1][0])

scenarios = {
    "Baseline": {},
    "Lower capacity pressure": {"capacity_utilization": .45},
    "Reduced staff shortage": {"staff_shortage_index": .15},
    "Improved digital access": {"digital_access_index": .90},
    "Reduced service delay": {"service_delay_hours": 1.0},
    "Combined improvement": {
        "capacity_utilization": .45,
        "staff_shortage_index": .15,
        "digital_access_index": .90,
        "service_delay_hours": 1.0,
        "resource_gap_index": .15
    }
}
scenario_results = []
for name, changes in scenarios.items():
    scenario_results.append({
        "scenario": name,
        "predicted_probability": scenario_prediction(base, changes)
    })
scenario_df = pd.DataFrame(scenario_results).sort_values("predicted_probability")
display(scenario_df.style.format({"predicted_probability": "{:.1%}"}))
plt.figure(figsize=(10,5))
sns.barplot(data=scenario_df, x="predicted_probability", y="scenario")
plt.xlim(0,1)
plt.xlabel("Predicted probability")
plt.title("Hypothetical scenario comparison")
plt.tight_layout()
plt.show()

## 14. Build a compact executive intelligence view

The dashboard combines model performance, predicted risk distribution, anomaly count, and top permutation-importance signals. It is a summary for exploration, not a production monitoring interface.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics[["roc_auc", "pr_auc", "f1"]].plot(kind="bar", ax=axes[0,0], ylim=(0,1))
axes[0,0].set_title("Model comparison")
axes[0,0].tick_params(axis="x", rotation=20)

sns.histplot(best_prob, bins=25, kde=True, ax=axes[0,1])
axes[0,1].set_title("Predicted risk-score distribution")
axes[0,1].set_xlabel("Predicted probability")

pd.Series(anomaly_flag).value_counts().sort_index().plot(
    kind="bar", ax=axes[1,0], color=["steelblue", "crimson"]
)
axes[1,0].set_title("Anomaly flags in test set")
axes[1,0].set_xlabel("0 = typical, 1 = unusual")

top_imp.tail(8).plot(x="feature", y="importance_mean", kind="barh",
                     legend=False, ax=axes[1,1])
axes[1,1].set_title("Top predictive signals")
axes[1,1].set_xlabel("Permutation importance")
plt.tight_layout()
plt.show()

## 15. Generate a concise findings report

This cell extracts the leading model and highest-ranked predictive signals from the actual run. Interpret these results in the context of the simulated data-generating process.

In [ ]:
top_features = importance.head(5)["feature"].tolist()
print("=" * 65)
print("AEGIS AI — EXPERIMENT SUMMARY")
print("=" * 65)
print("Records generated:", len(df))
print("Positive target rate:", f"{y.mean():.1%}")
print("Selected model:", best_name)
print("Test ROC-AUC:", f"{roc_auc_score(y_test, best_prob):.3f}")
print("Test PR-AUC:", f"{average_precision_score(y_test, best_prob):.3f}")
print("Test F1 at threshold 0.50:", f"{f1_score(y_test, best_pred):.3f}")
print("Anomalies flagged:", int(anomaly_flag.sum()), "of", len(anomaly_flag))
print("Top predictive signals:", ", ".join(top_features))
print("\nInterpretation:")
print("- Predictive performance is evaluated on a held-out synthetic test set.")
print("- Anomaly detection identifies unusual records, not necessarily harmful ones.")
print("- Permutation importance describes predictive contribution, not causation.")
print("- Scenario results are hypothetical model outputs, not guaranteed interventions.")

## 16. Export reusable outputs

The notebook exports the synthetic dataset, model comparison table, feature importance, and test-set predictions. Files are saved in the current working directory.

In [ ]:
predictions_export = X_test.copy()
predictions_export["actual_risk_event"] = y_test.to_numpy()
predictions_export["predicted_probability"] = best_prob
predictions_export["predicted_class_050"] = best_pred
predictions_export["anomaly_flag"] = anomaly_flag

df.to_csv("aegis_synthetic_dataset.csv", index=False)
metrics.to_csv("aegis_model_comparison.csv")
importance.to_csv("aegis_feature_importance.csv", index=False)
predictions_export.to_csv("aegis_test_predictions.csv", index=False)
print("Exported:")
for filename in [
    "aegis_synthetic_dataset.csv",
    "aegis_model_comparison.csv",
    "aegis_feature_importance.csv",
    "aegis_test_predictions.csv"
]:
    print(" •", filename)

## 17. Limitations, responsible use, and next steps

### Limitations
- All observations and labels are synthetic.
- The target was generated from a designed mathematical mechanism; real data may behave very differently.
- Random train/test splitting assumes observations are independent. A real temporal or geographic deployment should use time-aware or group-aware validation.
- Probability calibration, subgroup performance, fairness, and drift have not been established.
- The what-if simulator is not causal inference.

### Responsible next steps
1. Replace synthetic data with a legally obtained, documented real dataset.
2. Define the target with domain experts and validate label quality.
3. Use temporal and geographic holdouts to test generalization.
4. Calibrate probabilities and select thresholds with stakeholders.
5. Audit performance across regions and population groups.
6. Add monitoring for data drift, model drift, and human review.

### Final takeaway

**AEGIS AI demonstrates a reusable intelligence workflow:** prediction tells us what may happen, anomaly detection highlights what looks unusual, explainability helps inspect model behavior, and scenario analysis supports structured exploration. The strongest result is not a single score—it is a transparent process for turning data into questions worth investigating.